In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [2]:
# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../data/census/98-401-X2021002_eng_CSV/98-401-X2021002_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [3]:
df_ada_cma_rel = pd.read_csv('../data/census/ada_cma_relation.csv')
df_ada_cma_rel = df_ada_cma_rel.rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID', 'ADADGUID_ADAIDUGD': 'ADADGUID'})

In [4]:
gdf_cma = gpd.read_file('../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'PRUID', 'geometry']]

Load tariff information and join to CMAs

In [5]:
# Load ADA-level tariff counts and percents
df_tariffs_ada_count = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Counts').drop(columns=['geometry'])
df_tariffs_ada_pct = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Percents')

In [6]:
df_tariffs_count_filtered = (
    df_tariffs_ada_count
    # .drop(columns=['geometry'])
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Group by CMADGUID and sum all tariff columns
tariff_columns_count = [col for col in df_tariffs_count_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma_count = df_tariffs_count_filtered.groupby('CMADGUID')[tariff_columns_count].sum().reset_index()

print(f"Shape of CMA tariffs (counts) dataframe: {df_tariffs_cma_count.shape}")
df_tariffs_cma_count.head()

Shape of CMA tariffs (counts) dataframe: (152, 25)


,CMADGUID,Auto_B,Alum_B,Steel_B,Cop_B,Lum_B,Ene_B,CUSMA_B,Total_B,Auto_E,...,CUSMA_E,Total_E,Auto_C,Alum_C,Steel_C,Cop_C,Lum_C,Ene_C,CUSMA_C,Total_C
0,2021S0503001,7,17,14,2,8,10,118,118,40,...,1966,1966,146,226,216,23,110,247,2667,2667
1,2021S0503205,35,89,56,10,20,39,400,415,660,...,5596,6414,1252,2841,1673,137,676,1498,10111,11040
2,2021S0503305,15,37,34,5,8,19,177,191,316,...,3539,3919,378,684,598,102,344,387,4466,4725
3,2021S0503310,5,17,13,1,8,9,136,142,153,...,2460,3487,313,620,1401,22,365,1168,3691,4746
4,2021S0503320,9,26,15,0,9,10,126,132,172,...,1530,1696,191,389,309,1,215,223,2205,2357


In [ ]:
# Melt percents and counts to long format, extract base and numeric suffix
def melt_tariffs(df, value_name, suffix_map=None):
    df_long = df.melt(id_vars=['ADADGUID'], var_name='tariff', value_name=value_name)
    df_long['base'] = df_long['tariff'].str.replace(r'_(1|2|3|B|E|C)$', '', regex=True)
    df_long['suffix'] = df_long['tariff'].str.extract(r'_([1-3BEC])$')[0]
    if suffix_map:
        df_long['suffix'] = df_long['suffix'].map(suffix_map)
    return df_long

suffix_map_counts = {'B': '1', 'E': '2', 'C': '3'}

df_pct_long = melt_tariffs(df_tariffs_ada_pct, 'percent')
df_count_long = melt_tariffs(df_tariffs_ada_count, 'count', suffix_map=suffix_map_counts)

# Merge percents and counts, add CMA IDs
df_merge = (
    df_pct_long
    .merge(df_count_long, on=['ADADGUID', 'base', 'suffix'], how='left')
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Compute weighted percent and aggregate per CMA
df_cma = (
    df_merge.assign(weighted=lambda x: x['percent'] * x['count'])
    .groupby(['CMADGUID', 'base', 'suffix'], observed=True)
    .agg(total_weighted=('weighted', 'sum'), total_count=('count', 'sum'))
    .reset_index()
)
df_cma['cma_percent'] = df_cma['total_weighted'] / df_cma['total_count']
df_cma.loc[df_cma['total_count'] == 0, 'cma_percent'] = np.nan

# Pivot to wide format
df_cma['colname'] = df_cma['base'] + '_' + df_cma['suffix']
df_tariffs_cma_pct = df_cma.pivot(index='CMADGUID', columns='colname', values='cma_percent').reset_index()

print(f"Shape of CMA tariffs (percents) dataframe: {df_tariffs_cma_pct.shape}")
df_tariffs_cma_pct.head()

Shape of CMA tariffs (percents) dataframe: (152, 25)


colname,CMADGUID,Alum_1,Alum_2,Alum_3,Auto_1,Auto_2,Auto_3,CUSMA_1,CUSMA_2,CUSMA_3,...,Ene_3,Lum_1,Lum_2,Lum_3,Steel_1,Steel_2,Steel_3,Total_1,Total_2,Total_3
0,2021S0503001,0.005691,0.007065,0.002238,0.005103,0.002101,0.001510,0.021111,0.076244,0.025727,...,0.002480,0.004841,0.004708,0.001141,0.007445,0.008430,0.002153,0.021111,0.076244,0.025727
1,2021S0503205,0.012824,0.032155,0.012252,0.006093,0.012695,0.005186,0.060359,0.056219,0.043152,...,0.006926,0.009039,0.017713,0.009175,0.009748,0.016886,0.007021,0.061639,0.060594,0.046423
2,2021S0503305,0.011629,0.025271,0.008997,0.006637,0.008496,0.005317,0.055809,0.079371,0.061789,...,0.004941,0.005070,0.022372,0.004787,0.015260,0.024982,0.008183,0.060623,0.089119,0.064767
3,2021S0503310,0.010659,0.027720,0.011093,0.009016,0.033448,0.006249,0.108481,0.058331,0.070211,...,0.018699,0.017934,0.013391,0.007487,0.012512,0.081231,0.022146,0.107721,0.091723,0.082464
4,2021S0503320,0.013900,0.020012,0.008215,0.006885,0.026298,0.004083,0.065049,0.155038,0.052572,...,0.004378,0.023556,0.020492,0.011265,0.016517,0.025372,0.006948,0.066470,0.153233,0.055045


In [19]:
df_cen_cma_data_renamed = df_cen_cma_data.rename(columns={'DGUID': 'CMADGUID'})

# Merge census with CMA
df_final_counts = df_cen_cma_data_renamed.merge(df_tariffs_cma_count, on='CMADGUID', how='inner')
df_final_percents = df_cen_cma_data_renamed.merge(df_tariffs_cma_pct, on='CMADGUID', how='inner')

print(f"Shape of final counts dataframe: {df_final_counts.shape}")
print(f"Shape of final percents dataframe: {df_final_percents.shape}")

Shape of final counts dataframe: (152, 28)
Shape of final percents dataframe: (152, 28)


In [27]:
# Helper function to save centroids as CSV with WKT
def save_centroids_csv(gdf, path):
    df_csv = gdf.copy()
    df_csv['geometry'] = df_csv.geometry.apply(lambda geom: geom.wkt)
    df_csv.to_csv(path, index=False)

# Prepare geometry data
gdf_cma_geom = gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'})

# ---- COUNTS ----
gdf_final_counts_full = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry'
)

# Centroids
gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids.geometry.centroid
gdf_cma_centroids = gdf_cma_centroids.set_crs(gdf_cma.crs).to_crs('EPSG:4326')

gdf_final_counts_centroids = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save counts
gdf_final_counts_full.to_file('../data/cma/cma_tariffs_counts_full_geometry.gpkg', driver='GPKG')
save_centroids_csv(gdf_final_counts_centroids, '../data/cma/cma_tariffs_counts_centroids.csv')

# ---- PERCENTS ----
gdf_final_percents_full = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry'
)

gdf_final_percents_centroids = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save percents
gdf_final_percents_full.to_file('../data/cma/cma_tariffs_percents_full_geometry.gpkg', driver='GPKG')
save_centroids_csv(gdf_final_percents_centroids, '../data/cma/cma_tariffs_percents_centroids.csv')

# Print shapes
print(f"Counts full geometry shape: {gdf_final_counts_full.shape}")
print(f"Percents full geometry shape: {gdf_final_percents_full.shape}")
print(f"Counts centroids shape: {gdf_final_counts_centroids.shape}")
print(f"Percents centroids shape: {gdf_final_percents_centroids.shape}")

/tmp/ipykernel_2125208/2244572891.py:4: UserWarning: Geometry column does not contain geometry.
  df_csv['geometry'] = df_csv.geometry.apply(lambda geom: geom.wkt)


Counts full geometry shape: (156, 29)
Percents full geometry shape: (156, 29)
Counts centroids shape: (156, 29)
Percents centroids shape: (156, 29)


/tmp/ipykernel_2125208/2244572891.py:4: UserWarning: Geometry column does not contain geometry.
  df_csv['geometry'] = df_csv.geometry.apply(lambda geom: geom.wkt)
